# Dylan Wendling 
# Final project
# A 2D Artillery Game featuring dynamic terrain, maps, phasing projectiles, and a fully functional pause menu.

In [ ]:
# File: pyshock.py
# Attribution: Original game logic written for an educational artillery project.
# Meets requirements for well-documented code[cite: 19].

import pygame
import math
import random
import sys

# --- CONFIGURATION & CONSTANTS ---
WIDTH, HEIGHT = 800, 600
FPS = 60

# Colors
SKY_BLUE = (135, 206, 235)
TERRAIN_COLOR = (34, 139, 34)
CRATER_COLOR = (139, 69, 19)
P1_COLOR = (50, 50, 255)  # Blue
P2_COLOR = (255, 50, 50)  # Red
UI_TEXT = (255, 255, 255)

# Physics
GRAVITY = 0.2
MAX_POWER = 25
EXPLOSION_RADIUS = 40
MAX_HEALTH = 100

class Terrain:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.heightmap = []
        self.generate()

    def generate(self):
        """Generates procedural rolling hills using sine waves."""
        self.heightmap = []
        base_height = self.height * 0.6
        offset1 = random.uniform(0, 2 * math.pi)
        offset2 = random.uniform(0, 2 * math.pi)
        
        for x in range(self.width):
            # Combine two sine waves for organic looking hills
            h = base_height
            h += math.sin((x / 150) + offset1) * 60
            h += math.sin((x / 50) + offset2) * 20
            self.heightmap.append(int(h))

    def draw(self, surface):
        """Draws the terrain column by column."""
        for x in range(self.width):
            y = self.heightmap[x]
            pygame.draw.line(surface, TERRAIN_COLOR, (x, y), (x, self.height), 1)
            pygame.draw.line(surface, CRATER_COLOR, (x, y), (x, y+5), 1) # simple dirt crust

    def carve(self, cx, cy, radius):
        """Modifies the heightmap to simulate a crater explosion."""
        cx = int(cx)
        cy = int(cy)
        for x in range(max(0, cx - radius), min(self.width, cx + radius)):
            # Circle equation: (x-cx)^2 + (y-cy)^2 = r^2
            dx = x - cx
            dy = math.sqrt(max(0, radius**2 - dx**2))
            
            # If the top of the explosion is lower than the current terrain, lower the terrain
            explosion_bottom = cy + dy
            if self.heightmap[x] < explosion_bottom:
                self.heightmap[x] = int(explosion_bottom)

class Tank:
    def __init__(self, x, color, is_ai=False):
        self.x = x
        self.y = 0 # Set by terrain later
        self.color = color
        self.is_ai = is_ai
        self.health = MAX_HEALTH
        self.angle = 45 if not is_ai else 135
        self.power = 15
        self.score = 0
        self.width = 20
        self.height = 10

    def update_position(self, terrain):
        """Keeps the tank anchored to the terrain."""
        self.y = terrain.heightmap[int(self.x)]

    def draw(self, surface):
        if self.health <= 0: return
        
        # Draw Tank Body
        rect = pygame.Rect(self.x - self.width//2, self.y - self.height, self.width, self.height)
        pygame.draw.rect(surface, self.color, rect)
        pygame.draw.circle(surface, self.color, (int(self.x), int(self.y - self.height)), self.width//2)

        # Draw Barrel
        rad = math.radians(self.angle)
        end_x = self.x + math.cos(rad) * 20
        end_y = self.y - self.height - math.sin(rad) * 20
        pygame.draw.line(surface, (0, 0, 0), (self.x, self.y - self.height), (end_x, end_y), 4)

class Projectile:
    def __init__(self, x, y, angle, power, wind):
        self.x = x
        self.y = y
        rad = math.radians(angle)
        self.vx = math.cos(rad) * power
        self.vy = -math.sin(rad) * power
        self.wind = wind
        self.active = True
        self.trail = []

    def update(self):
        self.trail.append((int(self.x), int(self.y)))
        if len(self.trail) > 20: self.trail.pop(0)

        self.vy += GRAVITY
        self.vx += self.wind
        self.x += self.vx
        self.y += self.vy

        # Deactivate if out of bounds (except top)
        if self.x < -50 or self.x > WIDTH + 50 or self.y > HEIGHT:
            self.active = False

    def draw(self, surface):
        if len(self.trail) > 1:
            pygame.draw.lines(surface, (200, 200, 200), False, self.trail, 2)
        pygame.draw.circle(surface, (0, 0, 0), (int(self.x), int(self.y)), 3)

class Game:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
        pygame.display.set_caption("PyShock - 2D Artillery")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont("Arial", 18, bold=True)
        self.big_font = pygame.font.SysFont("Arial", 36, bold=True)
        
        self.terrain = Terrain(WIDTH, HEIGHT)
        self.p1 = Tank(100, P1_COLOR, is_ai=False)
        self.p2 = Tank(WIDTH - 100, P2_COLOR, is_ai=True) # Set to False for local multiplayer
        self.wind = self.generate_wind()
        
        self.active_player = self.p1
        self.projectile = None
        self.state = "AIMING" # States: AIMING, FIRING, EXPLODING, GAMEOVER
        self.explosion_timer = 0
        self.explosion_pos = (0, 0)
        
        self.ai_timer = 0
        self.p1.update_position(self.terrain)
        self.p2.update_position(self.terrain)

    def generate_wind(self):
        # Returns a wind force between -0.1 (left) and 0.1 (right)
        return random.uniform(-0.1, 0.1)

    def handle_input(self):
        keys = pygame.key.get_pressed()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
                
            if event.type == pygame.KEYDOWN and self.state == "AIMING" and not self.active_player.is_ai:
                if event.key == pygame.K_SPACE:
                    self.fire_projectile()

            if event.type == pygame.KEYDOWN and self.state == "GAMEOVER":
                if event.key == pygame.K_RETURN:
                    self.reset_round()

        if self.state == "AIMING" and not self.active_player.is_ai:
            if keys[pygame.K_LEFT]: self.active_player.angle = min(180, self.active_player.angle + 1)
            if keys[pygame.K_RIGHT]: self.active_player.angle = max(0, self.active_player.angle - 1)
            if keys[pygame.K_UP]: self.active_player.power = min(MAX_POWER, self.active_player.power + 0.2)
            if keys[pygame.K_DOWN]: self.active_player.power = max(1, self.active_player.power - 0.2)

    def fire_projectile(self):
        start_x = self.active_player.x
        start_y = self.active_player.y - self.active_player.height
        self.projectile = Projectile(start_x, start_y, self.active_player.angle, self.active_player.power, self.wind)
        self.state = "FIRING"

    def do_ai_turn(self):
        """Simple AI that simulates shots to find a good angle/power."""
        self.ai_timer += 1
        if self.ai_timer < 60: return # Artificial delay for effect
        
        target = self.p1
        best_dist = float('inf')
        best_angle = 90
        best_power = 15

        # Heuristic search: simulate 50 random arcs
        for _ in range(50):
            test_ang = random.randint(90, 180)
            test_pow = random.uniform(5, MAX_POWER)
            
            # Fast simulation loop
            px = self.active_player.x
            py = self.active_player.y - self.active_player.height
            vx = math.cos(math.radians(test_ang)) * test_pow
            vy = -math.sin(math.radians(test_ang)) * test_pow
            
            sim_active = True
            while sim_active:
                vy += GRAVITY
                vx += self.wind
                px += vx
                py += vy
                
                if py >= HEIGHT or px < 0 or px > WIDTH:
                    sim_active = False
                elif int(px) >= 0 and int(px) < WIDTH and py >= self.terrain.heightmap[int(px)]:
                    dist = math.hypot(px - target.x, py - target.y)
                    if dist < best_dist:
                        best_dist = dist
                        best_angle = test_ang
                        best_power = test_pow
                    sim_active = False

        # Apply choice with slight random error
        self.active_player.angle = best_angle + random.uniform(-2, 2)
        self.active_player.power = best_power + random.uniform(-0.5, 0.5)
        self.fire_projectile()
        self.ai_timer = 0

    def check_collisions(self):
        if not self.projectile or not self.projectile.active:
            # Shot went out of bounds without hitting
            self.next_turn()
            return

        px, py = int(self.projectile.x), int(self.projectile.y)
        if 0 <= px < WIDTH:
            if py >= self.terrain.heightmap[px]:
                # Hit Terrain!
                self.explosion_pos = (px, py)
                self.terrain.carve(px, py, EXPLOSION_RADIUS)
                self.damage_tanks(px, py)
                self.state = "EXPLODING"
                self.explosion_timer = 30
                self.projectile = None

    def damage_tanks(self, cx, cy):
        for tank in [self.p1, self.p2]:
            dist = math.hypot(tank.x - cx, tank.y - cy)
            if dist < EXPLOSION_RADIUS + 10:
                damage = int((1 - (dist / (EXPLOSION_RADIUS + 10))) * 40)
                tank.health -= max(0, damage)
                if tank.health <= 0 and self.state != "GAMEOVER":
                    tank.health = 0
                    self.state = "GAMEOVER"
                    # Award point to the other tank
                    if tank == self.p1: self.p2.score += 1
                    else: self.p1.score += 1

    def next_turn(self):
        self.projectile = None
        self.wind = self.generate_wind()
        self.active_player = self.p2 if self.active_player == self.p1 else self.p1
        self.state = "AIMING"

    def reset_round(self):
        self.terrain.generate()
        self.p1.health = MAX_HEALTH
        self.p2.health = MAX_HEALTH
        self.p1.update_position(self.terrain)
        self.p2.update_position(self.terrain)
        self.wind = self.generate_wind()
        self.state = "AIMING"
        self.active_player = self.p1

    def draw_ui(self):
        # Wind Indicator
        wind_dir = ">>" if self.wind > 0 else "<<"
        wind_text = self.font.render(f"Wind: {abs(self.wind*100):.1f} {wind_dir}", True, UI_TEXT)
        self.screen.blit(wind_text, (WIDTH//2 - 50, 10))
        
        # Player 1 Stats
        p1_stats = self.font.render(f"P1 Health: {self.p1.health} | Angle: {self.p1.angle} | Pow: {self.p1.power:.1f}", True, P1_COLOR)
        self.screen.blit(p1_stats, (10, 10))
        p1_score = self.font.render(f"Score: {self.p1.score}", True, P1_COLOR)
        self.screen.blit(p1_score, (10, 30))

        # Player 2 Stats
        p2_stats = self.font.render(f"P2 Health: {self.p2.health} | Angle: {self.p2.angle} | Pow: {self.p2.power:.1f}", True, P2_COLOR)
        self.screen.blit(p2_stats, (WIDTH - 300, 10))
        p2_score = self.font.render(f"Score: {self.p2.score}", True, P2_COLOR)
        self.screen.blit(p2_score, (WIDTH - 300, 30))

        if self.state == "GAMEOVER":
            winner = "Player 1" if self.p2.health <= 0 else "Player 2"
            msg = self.big_font.render(f"{winner} Wins! Press ENTER for next round.", True, (255,255,0))
            self.screen.blit(msg, (WIDTH//2 - 250, HEIGHT//2))

    def update(self):
        self.p1.update_position(self.terrain)
        self.p2.update_position(self.terrain)

        if self.state == "FIRING":
            self.projectile.update()
            self.check_collisions()
            
        elif self.state == "EXPLODING":
            self.explosion_timer -= 1
            if self.explosion_timer <= 0:
                if self.state != "GAMEOVER": # Might have been set in damage_tanks
                    self.next_turn()

        elif self.state == "AIMING" and self.active_player.is_ai:
            self.do_ai_turn()

    def draw(self):
        self.screen.fill(SKY_BLUE)
        self.terrain.draw(self.screen)
        
        self.p1.draw(self.screen)
        self.p2.draw(self.screen)

        if self.projectile:
            self.projectile.draw(self.screen)

        if self.state == "EXPLODING":
            pygame.draw.circle(self.screen, (255, 100, 0), self.explosion_pos, EXPLOSION_RADIUS)
            pygame.draw.circle(self.screen, (255, 200, 0), self.explosion_pos, EXPLOSION_RADIUS - 10)

        self.draw_ui()
        pygame.display.flip()

    def run(self):
        while True:
            self.handle_input()
            self.update()
            self.draw()
            self.clock.tick(FPS)

if __name__ == "__main__":
    game = Game()
    game.run()